Without tool calling our local model has been trapped in a box. It only knows what it learned during training. If you ask it for the current weather, the latest stock price, or to turn on your smart lights, it can't do it.

Tool calling (or function calling) gives your LLM "hands."  It allows you to describe a Python function to the model, and if the model decides it needs that function to answer your prompt, it won't reply with text. Instead, it will reply with a structured request to run your code!

In [2]:
pip install ollama

  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Defining a tool

import ollama

# 1. Define our tool
def get_flight_time(departure: str, destination: str) -> str:
    """Get the estimated flight time between two cities."""
    # In a real app, this would ping a flight API. 
    # For now, we will just mock a response.
    return f"The flight from {departure} to {destination} is approximately 4 hours."

In [5]:
# 2. Ask a question that requires the tool
response = ollama.chat(
    model='llama3.1',
    messages=[{'role': 'user', 'content': 'What is the capital of Jamaica? Also, how long is the flight from New York to Kingston?'}],
    tools=[get_flight_time] # <-- We give the model access to our function here
)

# 3. Check how the model responded
message = response['message']

if message.get('tool_calls'):
    print("🤖 The model decided to use a tool!")
    for tool in message['tool_calls']:
        print(f"Tool Name: {tool['function']['name']}")
        print(f"Arguments: {tool['function']['arguments']}")
else:
    print("🤖 The model answered directly:")
    print(message['content'])
    
# it will only respond to one of the questions, to solve this you need multi-intent routing

🤖 The model decided to use a tool!
Tool Name: get_flight_time
Arguments: {'departure': 'New York', 'destination': 'Kingston'}


you essentially write a loop to parse a tool call and manually run the loop and append it such that the model think it has already ran the tool call?

In [10]:
messages=[{'role': 'user', 'content': 'What is the capital of Jamaica? Also, how long is the flight from New York to Kingston?'}]
print("1a. Sending initial prompt with multi-intent routing...")
response = ollama.chat(model='llama3.1', messages=messages, tools=[get_flight_time])
message = response['message']
print("1b. Response from initial message:", message)

# 2. The Loop: Did the model ask for a tool?
if message.get('tool_calls'):
    print("2. 🤖 Model paused to call a tool!")
    
    # Add the model's tool request to our message history
    messages.append(message)
    
    # 3. Execute the tools
    for tool in message['tool_calls']:
        if tool['function']['name'] == 'get_flight_time':
            # Extract arguments and run our actual Python function
            args = tool['function']['arguments']
            flight_result = get_flight_time(args['departure'], args['destination'])
            
            print(f"3a. ⚙️ Running function: get_flight_time -> Result: {flight_result}")
            
            # Add the function's result back into the history as a 'tool' message
            messages.append({
                'role': 'tool',
                'content': flight_result,
            })
            
            print("3b. new message appended with tool call", messages)
    
    # 4. Send the updated history back to the model for a final answer
    print("4. Sending tool results back to the model for a final answer...")
    final_response = ollama.chat(model='llama3.1', messages=messages)
    print("\n🤖 Final Output:\n", final_response['message']['content'])

else:
    print("\n🤖 Direct Output:\n", message['content'])

1a. Sending initial prompt with multi-intent routing...
1b. Response from initial message: role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='get_flight_time', arguments={'departure': 'New York', 'destination': 'Kingston'}))]
2. 🤖 Model paused to call a tool!
3a. ⚙️ Running function: get_flight_time -> Result: The flight from New York to Kingston is approximately 4 hours.
3b. new message appended with tool call [{'role': 'user', 'content': 'What is the capital of Jamaica? Also, how long is the flight from New York to Kingston?'}, Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='get_flight_time', arguments={'departure': 'New York', 'destination': 'Kingston'}))]), {'role': 'tool', 'content': 'The flight from New York to Kingston is approximately 4 hours.'}]
4. Sending tool results back to the model for a final answer...

🤖 Final Output:
 The capital